# 08 — Predictive Analysis: Classification

Modèles comparés sur la même séparation test ; hyperparamètres choisis par validation croisée sur TRAIN seulement. Scores non préremplis.

Source : jeu historique IBM DS0321EN. Exécuter toutes les cellules dans Google Colab. Les sorties sont produites à l’exécution, pas préremplies.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
BASE="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/"
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)
data=pd.read_csv(BASE+'dataset_part_2.csv')
features=pd.read_csv(BASE+'dataset_part_3.csv')
# Remove a CSV index column if present
features=features.loc[:,~features.columns.str.match(r'^Unnamed:')]
assert len(data)==len(features), 'Dataset mismatch; check IBM sources'
assert 'Class' in data.columns
X=features.apply(pd.to_numeric,errors='raise')
y=data['Class'].astype(int)
print('Shape:',X.shape,'labels:',y.value_counts().to_dict())
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=2,stratify=y)
print('Training/test:',len(y_train),len(y_test))

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=2)
models={
 'Logistic Regression':(Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(max_iter=3000))]),{'model__C':[.01,.1,1,10]}),
 'SVM':(Pipeline([('scaler',StandardScaler()),('model',SVC())]),{'model__kernel':['linear','rbf','sigmoid'],'model__C':[.1,1,10]}),
 'Decision Tree':(DecisionTreeClassifier(random_state=2),{'max_depth':[2,3,4,6,8,None],'min_samples_leaf':[1,2,4]}),
 'KNN':(Pipeline([('scaler',StandardScaler()),('model',KNeighborsClassifier())]),{'model__n_neighbors':[3,5,7,9]})
}
results=[]
for name,(est,params) in models.items():
    search=GridSearchCV(est,params,cv=cv,scoring='accuracy',n_jobs=-1)
    search.fit(X_train,y_train)
    pred=search.predict(X_test)
    results.append({'Model':name,'ValidationAccuracy':search.best_score_,
                    'TestAccuracy':accuracy_score(y_test,pred),
                    'BestParams':str(search.best_params_)})
    cm=confusion_matrix(y_test,pred,labels=[0,1]);disp=ConfusionMatrixDisplay(cm,display_labels=['Failure','Success'])
    disp.plot(cmap='Blues',colorbar=False)
    plt.title(name); plt.tight_layout();plt.savefig(OUT/('confusion_'+name.replace(' ','_')+'.png'),dpi=160);plt.show()
    print(name,'\n',classification_report(y_test,pred,zero_division=0))
results_df=pd.DataFrame(results).sort_values('ValidationAccuracy',ascending=False)
display(results_df);results_df.to_csv(OUT/'model_comparison.csv',index=False)


In [ ]:
ax=results_df.plot(x='Model',y=['ValidationAccuracy','TestAccuracy'],kind='bar',figsize=(10,5),ylim=(0,1))
plt.ylabel('Accuracy');plt.title('Validation vs Test performance');plt.xticks(rotation=20);plt.tight_layout()
plt.savefig(OUT/'model_comparison.png',dpi=180);plt.show()
print('Best by validation:',results_df.iloc[0]['Model'])
print('Note: selecting the winner using test accuracy would create test-selection bias.')
import zipfile
with zipfile.ZipFile('08_ML_outputs.zip','w') as z:
 for p in OUT.iterdir(): z.write(p,arcname=p.name)
from google.colab import files
files.download('08_ML_outputs.zip')